# Module 10 • Advanced Applications
# Lesson 60 • Conversational AI and Dialogue Systems — Intent, State Tracking, Response Generation, Memory, and Evaluation

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Execution target:** CPU only

## Scope

This lesson builds a complete conversational AI pipeline.

It covers:

- task-oriented dialogue;
- open-domain dialogue;
- intent classification;
- slot extraction;
- dialogue state tracking;
- dialogue policy;
- response generation;
- short-term memory;
- session context;
- fallback and abstention;
- multi-turn evaluation;
- task success;
- intent accuracy;
- slot accuracy;
- response relevance;
- safety and access control;
- Arabic and multilingual dialogue.

The executable system is fully offline and deterministic.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish task-oriented and open-domain dialogue;
- classify user intents;
- extract slot values;
- maintain dialogue state;
- define dialogue policies;
- generate context-aware responses;
- implement short-term memory;
- handle unknown or unsupported requests;
- evaluate multi-turn conversations;
- distinguish intent, state, and response errors;
- design multilingual and Arabic-aware dialogue systems;
- map a notebook prototype to a production conversational architecture.

## Table of Contents

1. Conversational AI Overview  
2. Dialogue System Types  
3. Task-Oriented Dialogue  
4. Open-Domain Dialogue  
5. Dialogue Turn Structure  
6. Intent Recognition  
7. Slot Filling  
8. Dialogue State Tracking  
9. Dialogue Policy  
10. Natural Language Generation  
11. Memory  
12. Session Management  
13. Offline Intent Dataset  
14. Intent Classifier  
15. Intent Prediction  
16. Slot Extraction  
17. State Representation  
18. State Update  
19. Policy Rules  
20. Response Templates  
21. Dialogue Manager  
22. Multi-Turn Conversation  
23. Context Carry-Over  
24. Ellipsis and Follow-Up Turns  
25. Confirmation  
26. Clarification  
27. Fallback  
28. Abstention  
29. Memory Window  
30. User Preferences  
31. Dialogue Evaluation Dataset  
32. Intent Accuracy  
33. Slot Accuracy  
34. State Accuracy  
35. Task Success  
36. Response Relevance  
37. Turn Efficiency  
38. Error Taxonomy  
39. Intent Errors  
40. State Errors  
41. Policy Errors  
42. Response Errors  
43. Multilingual Dialogue  
44. Arabic Dialogue  
45. Tashkeel Policy  
46. Safety  
47. Privacy  
48. Production Architecture  
49. Latency  
50. Monitoring  
51. Optional LLM Dialogue Template  
52. Reproducibility  
53. Knowledge Check  
54. Exercises  
55. Summary and Next Lesson

# 1. Conversational AI Overview

Conversational AI systems process a sequence of turns rather than isolated inputs.

A dialogue system must reason over:

- the current utterance;
- previous turns;
- user goals;
- slot values;
- system actions;
- unresolved questions;
- safety constraints.

In [ ]:
import platform
import random
import re
import time
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

dialogue_types = pd.DataFrame(
    [
        ("Task-oriented", "complete a concrete user goal", "booking, support, search"),
        ("Open-domain", "free conversation", "general assistants"),
        ("Retrieval-based", "select known response/evidence", "FAQ/chat support"),
        ("Generative", "produce new response text", "LLM assistants"),
    ],
    columns=["Type", "Objective", "Example"],
)

dialogue_types

# 2. Dialogue System Types

Dialogue systems differ in how constrained the user goal is.

Task-oriented systems usually have explicit intents and slots. Open-domain systems need
broader language understanding and stronger generation controls.

# 3. Task-Oriented Dialogue

A task-oriented dialogue might represent:

```text
intent = book_flight
slots = {
    origin: Cairo,
    destination: London,
    date: tomorrow
}
```

The system tracks which required values are known and which remain missing.

# 4. Open-Domain Dialogue

Open-domain dialogue is less structured.

Its main challenges include:

- coherence;
- long-range memory;
- grounding;
- safety;
- persona consistency;
- factuality.

# 5. Dialogue Turn Structure

A typical task-oriented turn:

```text
user utterance
   ↓
NLU
   ├── intent
   └── slots
   ↓
dialogue state
   ↓
policy
   ↓
system action
   ↓
response generation
```

# 6. Intent Recognition

Intent classification predicts the user's current goal.

Example intents in this notebook:

- `greet`;
- `book_flight`;
- `weather_query`;
- `cancel_booking`;
- `goodbye`.

# 7. Slot Filling

Slots are structured values associated with an intent.

For flight booking:

- origin;
- destination;
- date.

# 8. Dialogue State Tracking

Dialogue state accumulates information across turns.

Example:

```text
turn 1: destination = London
turn 2: date = tomorrow
turn 3: origin = Cairo
```

The system should preserve earlier values unless the user changes them.

# 9. Dialogue Policy

A policy decides what the system should do next.

Possible actions:

- answer directly;
- ask for a missing slot;
- confirm a risky action;
- retrieve information;
- execute a tool;
- abstain.

# 10. Natural Language Generation

Response generation maps a system action into user-visible language.

A production system may use:

- templates;
- retrieval;
- seq2seq models;
- LLM generation.

# 11. Memory

Memory can be separated into:

- **turn memory** — recent conversation;
- **session state** — structured task variables;
- **long-term memory** — durable user preferences or profile information.

These should not be conflated.

# 12. Session Management

Each conversation should have explicit state.

This allows:

- reset;
- handoff;
- concurrent sessions;
- reproducible debugging.

# 13. Offline Intent Dataset

In [ ]:
intent_examples = [
    ("hello", "greet"),
    ("hi", "greet"),
    ("good morning", "greet"),
    ("hey there", "greet"),

    ("book a flight", "book_flight"),
    ("i need a flight", "book_flight"),
    ("reserve a ticket to london", "book_flight"),
    ("i want to fly tomorrow", "book_flight"),

    ("what is the weather", "weather_query"),
    ("weather in cairo", "weather_query"),
    ("will it rain tomorrow", "weather_query"),
    ("temperature in london", "weather_query"),

    ("cancel my booking", "cancel_booking"),
    ("cancel the reservation", "cancel_booking"),
    ("i want to cancel my flight", "cancel_booking"),
    ("remove my booking", "cancel_booking"),

    ("bye", "goodbye"),
    ("goodbye", "goodbye"),
    ("see you later", "goodbye"),
    ("thanks bye", "goodbye"),
]

intent_frame = pd.DataFrame(
    intent_examples,
    columns=["utterance", "intent"],
)

intent_frame

# 14. Intent Classifier

In [ ]:
intent_model = make_pipeline(
    TfidfVectorizer(
        ngram_range=(1, 2),
        lowercase=True,
    ),
    LogisticRegression(
        max_iter=1000,
        random_state=SEED,
    ),
)

intent_model.fit(
    intent_frame["utterance"],
    intent_frame["intent"],
)

print("Intent classes:", list(intent_model.classes_))

# 15. Intent Prediction

In [ ]:
def predict_intent(
    utterance,
):
    probabilities = intent_model.predict_proba(
        [utterance]
    )[0]

    best_index = int(
        probabilities.argmax()
    )

    return {
        "intent": intent_model.classes_[
            best_index
        ],
        "confidence": float(
            probabilities[
                best_index
            ]
        ),
    }


predict_intent(
    "i need to reserve a flight"
)

# 16. Slot Extraction

This educational slot extractor uses transparent regular expressions and gazetteers.

Production systems may use token classification, span extraction, or LLM-based parsing.

In [ ]:
CITIES = [
    "cairo",
    "london",
    "paris",
    "new york",
    "knoxville",
]

DATE_PATTERNS = {
    "today": "today",
    "tomorrow": "tomorrow",
    "next week": "next week",
    "monday": "monday",
    "tuesday": "tuesday",
}

def extract_slots(
    utterance,
):
    text = utterance.lower()

    found_cities = [
        city
        for city in CITIES
        if city in text
    ]

    slots = {}

    to_match = re.search(
        r"\bto\s+([a-z ]+)",
        text,
    )

    from_match = re.search(
        r"\bfrom\s+([a-z ]+)",
        text,
    )

    if from_match:
        phrase = from_match.group(1)
        for city in CITIES:
            if city in phrase:
                slots[
                    "origin"
                ] = city
                break

    if to_match:
        phrase = to_match.group(1)
        for city in CITIES:
            if city in phrase:
                slots[
                    "destination"
                ] = city
                break

    if (
        len(
            found_cities
        )
        == 1
        and "destination"
        not in slots
        and "origin"
        not in slots
    ):
        slots[
            "destination"
        ] = found_cities[
            0
        ]

    for phrase, value in DATE_PATTERNS.items():
        if phrase in text:
            slots[
                "date"
            ] = value
            break

    return slots


extract_slots(
    "book a flight from cairo to london tomorrow"
)

# 17. State Representation

In [ ]:
def new_dialogue_state():
    return {
        "active_intent": None,
        "slots": {
            "origin": None,
            "destination": None,
            "date": None,
        },
        "last_system_action": None,
        "turn_count": 0,
        "completed": False,
    }


state = new_dialogue_state()
state

# 18. State Update

In [ ]:
def update_state(
    state,
    intent_result,
    extracted_slots,
):
    new_state = {
        "active_intent": (
            state[
                "active_intent"
            ]
        ),
        "slots": dict(
            state[
                "slots"
            ]
        ),
        "last_system_action": (
            state[
                "last_system_action"
            ]
        ),
        "turn_count": (
            state[
                "turn_count"
            ]
            + 1
        ),
        "completed": False,
    }

    predicted_intent = (
        intent_result[
            "intent"
        ]
    )

    if predicted_intent not in {
        "greet",
        "goodbye",
    }:
        new_state[
            "active_intent"
        ] = predicted_intent

    for slot_name, value in (
        extracted_slots.items()
    ):
        if slot_name in new_state[
            "slots"
        ]:
            new_state[
                "slots"
            ][
                slot_name
            ] = value

    return new_state

# 19. Policy Rules

In [ ]:
REQUIRED_BOOKING_SLOTS = [
    "origin",
    "destination",
    "date",
]

def dialogue_policy(
    state,
    intent_result,
):
    intent = intent_result[
        "intent"
    ]

    if (
        intent_result[
            "confidence"
        ]
        < 0.35
    ):
        return {
            "action": "clarify_intent"
        }

    if intent == "greet":
        return {
            "action": "greet"
        }

    if intent == "goodbye":
        return {
            "action": "goodbye"
        }

    if intent == "weather_query":
        return {
            "action": "weather_not_connected"
        }

    if intent == "cancel_booking":
        return {
            "action": "confirm_cancellation"
        }

    if (
        state[
            "active_intent"
        ]
        == "book_flight"
    ):
        missing = [
            slot
            for slot in REQUIRED_BOOKING_SLOTS
            if not state[
                "slots"
            ][
                slot
            ]
        ]

        if missing:
            return {
                "action": "request_slot",
                "slot": missing[
                    0
                ],
            }

        return {
            "action": "confirm_booking",
        }

    return {
        "action": "fallback"
    }

# 20. Response Templates

In [ ]:
def render_response(
    action,
    state,
):
    action_name = action[
        "action"
    ]

    if action_name == "greet":
        return "Hello. How can I help you?"

    if action_name == "goodbye":
        return "Goodbye."

    if action_name == "clarify_intent":
        return (
            "I am not confident I understood the request. "
            "Could you rephrase it?"
        )

    if action_name == "request_slot":
        slot = action[
            "slot"
        ]

        prompts = {
            "origin": (
                "What city are you departing from?"
            ),
            "destination": (
                "What is your destination?"
            ),
            "date": (
                "What date would you like to travel?"
            ),
        }

        return prompts[
            slot
        ]

    if action_name == "confirm_booking":
        slots = state[
            "slots"
        ]

        return (
            "Please confirm the flight request "
            f"from {slots['origin']} "
            f"to {slots['destination']} "
            f"on {slots['date']}."
        )

    if action_name == "confirm_cancellation":
        return (
            "Please confirm that you want to cancel the booking."
        )

    if action_name == "weather_not_connected":
        return (
            "Weather lookup is not connected in this offline lesson."
        )

    return (
        "I cannot complete that request with the current offline dialogue system."
    )

# 21. Dialogue Manager

In [ ]:
class DialogueManager:
    def __init__(
        self,
        memory_size=6,
    ):
        self.state = (
            new_dialogue_state()
        )

        self.memory = deque(
            maxlen=memory_size
        )

    def process(
        self,
        user_utterance,
    ):
        intent_result = (
            predict_intent(
                user_utterance
            )
        )

        slots = extract_slots(
            user_utterance
        )

        self.state = update_state(
            self.state,
            intent_result,
            slots,
        )

        action = dialogue_policy(
            self.state,
            intent_result,
        )

        response = render_response(
            action,
            self.state,
        )

        self.state[
            "last_system_action"
        ] = action[
            "action"
        ]

        self.memory.append({
            "user": user_utterance,
            "intent": intent_result[
                "intent"
            ],
            "slots": slots,
            "system": response,
        })

        return {
            "intent": (
                intent_result
            ),
            "slots": slots,
            "state": {
                "active_intent": (
                    self.state[
                        "active_intent"
                    ]
                ),
                "slots": dict(
                    self.state[
                        "slots"
                    ]
                ),
                "turn_count": (
                    self.state[
                        "turn_count"
                    ]
                ),
                "last_system_action": (
                    self.state[
                        "last_system_action"
                    ]
                ),
            },
            "response": (
                response
            ),
        }

# 22. Multi-Turn Conversation

In [ ]:
dialogue = DialogueManager()

conversation = [
    "hello",
    "i need a flight to london",
    "from cairo",
    "tomorrow",
]

conversation_rows = []

for utterance in conversation:
    result = dialogue.process(
        utterance
    )

    conversation_rows.append({
        "user": utterance,
        "intent": result[
            "intent"
        ][
            "intent"
        ],
        "slots_extracted": (
            result[
                "slots"
            ]
        ),
        "state": (
            result[
                "state"
            ][
                "slots"
            ]
        ),
        "system": (
            result[
                "response"
            ]
        ),
    })

pd.DataFrame(
    conversation_rows
)

# 23. Context Carry-Over

The dialogue manager keeps structured slot values across turns.

This is more reliable than trying to reconstruct every fact from raw conversation text
on every turn.

# 24. Ellipsis and Follow-Up Turns

Users often say:

- "tomorrow";
- "from Cairo";
- "yes";
- "the second one".

These utterances depend on prior context and cannot be interpreted correctly in
isolation.

# 25. Confirmation

Confirmation should be used before:

- irreversible actions;
- purchases;
- cancellations;
- high-cost operations;
- sensitive changes.

# 26. Clarification

Clarification is appropriate when:

- intent confidence is low;
- required slots conflict;
- the user provides ambiguous values.

# 27. Fallback

Fallback behavior should be explicit and safe.

A good fallback does not invent capabilities that the system does not have.

# 28. Abstention

Abstention means choosing not to provide an uncertain answer or action.

This is especially important when the dialogue system might otherwise perform an
incorrect transaction.

# 29. Memory Window

In [ ]:
memory_example = DialogueManager(
    memory_size=3
)

for utterance in [
    "hello",
    "book a flight to paris",
    "from cairo",
    "tomorrow",
]:
    memory_example.process(
        utterance
    )

list(
    memory_example.memory
)

# 30. User Preferences

Long-term preferences should be stored separately from transient task state.

Examples:

- preferred language;
- default airport;
- accessibility preference.

A system should not treat every transient slot value as a permanent preference.

# 31. Dialogue Evaluation Dataset

In [ ]:
intent_test = [
    ("hello there", "greet"),
    ("please reserve a flight", "book_flight"),
    ("weather tomorrow", "weather_query"),
    ("cancel the ticket", "cancel_booking"),
    ("goodbye for now", "goodbye"),
]

intent_test_frame = pd.DataFrame(
    intent_test,
    columns=[
        "utterance",
        "reference_intent",
    ],
)

intent_test_frame

# 32. Intent Accuracy

In [ ]:
intent_predictions = []

for row in intent_test_frame.itertuples(
    index=False
):
    prediction = predict_intent(
        row.utterance
    )

    intent_predictions.append({
        "utterance": (
            row.utterance
        ),
        "reference": (
            row.reference_intent
        ),
        "prediction": (
            prediction[
                "intent"
            ]
        ),
        "correct": (
            prediction[
                "intent"
            ]
            == row.reference_intent
        ),
        "confidence": (
            prediction[
                "confidence"
            ]
        ),
    })

intent_eval = pd.DataFrame(
    intent_predictions
)

intent_eval

In [ ]:
intent_accuracy = float(
    intent_eval[
        "correct"
    ].mean()
)

intent_accuracy

# 33. Slot Accuracy

In [ ]:
slot_tests = [
    (
        "flight from cairo to london tomorrow",
        {
            "origin": "cairo",
            "destination": "london",
            "date": "tomorrow",
        },
    ),
    (
        "to paris next week",
        {
            "destination": "paris",
            "date": "next week",
        },
    ),
]

slot_rows = []

for utterance, reference_slots in (
    slot_tests
):
    predicted_slots = (
        extract_slots(
            utterance
        )
    )

    keys = set(
        reference_slots
    ) | set(
        predicted_slots
    )

    correct = sum(
        reference_slots.get(
            key
        )
        == predicted_slots.get(
            key
        )
        for key in keys
    )

    slot_rows.append({
        "utterance": utterance,
        "reference": (
            reference_slots
        ),
        "prediction": (
            predicted_slots
        ),
        "slot_accuracy": (
            correct
            / len(
                keys
            )
            if keys
            else 1.0
        ),
    })

slot_eval = pd.DataFrame(
    slot_rows
)

slot_eval

# 34. State Accuracy

In [ ]:
state_test_dialogue = DialogueManager()

state_test_dialogue.process(
    "book a flight to london"
)

state_test_dialogue.process(
    "from cairo"
)

final_result = state_test_dialogue.process(
    "tomorrow"
)

expected_state = {
    "origin": "cairo",
    "destination": "london",
    "date": "tomorrow",
}

state_accuracy = float(
    final_result[
        "state"
    ][
        "slots"
    ]
    == expected_state
)

state_accuracy

# 35. Task Success

In [ ]:
def task_success(
    state,
):
    return float(
        state[
            "active_intent"
        ]
        == "book_flight"
        and all(
            state[
                "slots"
            ][
                slot
            ]
            for slot in REQUIRED_BOOKING_SLOTS
        )
    )


task_success(
    dialogue.state
)

# 36. Response Relevance

Response relevance asks whether the system action matches the current dialogue need.

This can be evaluated with:

- action labels;
- human judgments;
- semantic similarity;
- task completion.

# 37. Turn Efficiency

In [ ]:
def turn_efficiency(
    turns_used,
    minimum_required_turns,
):
    if turns_used <= 0:
        return 0.0

    return min(
        1.0,
        minimum_required_turns
        / turns_used,
    )


turn_efficiency(
    turns_used=4,
    minimum_required_turns=3,
)

# 38. Error Taxonomy

In [ ]:
error_taxonomy = pd.DataFrame(
    [
        ("Intent", "wrong user goal predicted"),
        ("Slot", "wrong or missing entity/value"),
        ("State", "previous information lost or corrupted"),
        ("Policy", "wrong next action selected"),
        ("Response", "action correct but wording poor"),
        ("Memory", "irrelevant or stale context used"),
        ("Safety", "unsafe or unauthorized action allowed"),
        ("Fallback", "system pretends to support an unavailable capability"),
    ],
    columns=[
        "Error type",
        "Description",
    ],
)

error_taxonomy

# 39. Intent Errors

Intent mistakes often cascade into:

- wrong slots;
- wrong policy;
- wrong response.

Therefore, error analysis should trace the earliest failing stage.

# 40. State Errors

State errors include:

- forgetting a slot;
- overwriting a slot incorrectly;
- confusing current and previous tasks.

# 41. Policy Errors

Policy can fail even when NLU is correct.

Example: the system knows all booking slots but asks for the destination again.

# 42. Response Errors

Response generation can fail through:

- contradiction;
- repetition;
- wrong tone;
- unsupported information;
- missing confirmation.

# 43. Multilingual Dialogue

Multilingual systems may use:

- multilingual intent classifiers;
- language-specific slot extraction;
- translation layers;
- multilingual LLMs;
- per-language response templates.

Each supported language should have its own evaluation data.

# 44. Arabic Dialogue

Arabic dialogue systems must consider:

- morphology;
- clitics;
- orthographic variation;
- optional tashkeel;
- dialect differences;
- Arabic/Latin code-switching.

# 45. Tashkeel Policy

In [ ]:
ARABIC_DIACRITICS = set(
    "\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652"
)

def strip_tashkeel(
    text,
):
    return "".join(
        character
        for character in text
        if character
        not in ARABIC_DIACRITICS
    )

arabic_utterance = (
    "أُرِيدُ حَجْزَ رِحْلَةٍ إِلَى لَنْدَنَ غَدًا."
)

pd.Series({
    "fully_vocalized": (
        arabic_utterance
    ),
    "diagnostic_without_tashkeel": (
        strip_tashkeel(
            arabic_utterance
        )
    ),
})

For a fully vocalized Arabic dialogue task, preserve tashkeel in user utterances,
training examples, generated responses, and primary evaluation.

# 46. Safety

A conversational system should require confirmation before sensitive actions and should
not claim that an external action succeeded unless it actually did.

# 47. Privacy

Dialogue memory may contain personal information.

Production systems need:

- retention policies;
- access control;
- redaction;
- session isolation;
- user-visible memory controls.

# 48. Production Architecture

```text
user message
   ↓
language detection / normalization
   ↓
NLU or LLM parser
   ↓
dialogue state
   ↓
policy / tool router
   ↓
tools, retrieval, business logic
   ↓
response generation
   ↓
safety + grounding checks
   ↓
response
```

# 49. Latency

In [ ]:
def measure_latency(
    dialogue_manager,
    utterance,
    repeats=100,
):
    durations = []

    for _ in range(
        repeats
    ):
        local_dialogue = DialogueManager()

        start = time.perf_counter()

        local_dialogue.process(
            utterance
        )

        durations.append(
            (
                time.perf_counter()
                - start
            )
            * 1000.0
        )

    return {
        "mean_ms": float(
            np.mean(
                durations
            )
        ),
        "p95_ms": float(
            np.percentile(
                durations,
                95,
            )
        ),
    }


measure_latency(
    DialogueManager(),
    "book a flight to london tomorrow",
)

# 50. Monitoring

Monitor:

- intent confidence;
- fallback rate;
- slot completion rate;
- task success;
- average turns per task;
- abandonment rate;
- clarification rate;
- unsafe-action blocks;
- latency;
- language distribution;
- memory-related errors.

# 51. Optional LLM Dialogue Template

This section is non-executable because it would require an external model.

A modern LLM-based dialogue system commonly uses:

```text
system instructions
  + recent turns
  + structured state
  + tool schemas
  + retrieved context
      ↓
LLM
      ↓
structured action or response
```

Even with an LLM, explicit state, confirmation rules, tool permissions, and evaluation
remain useful.

# 52. Reproducibility

In [ ]:
pd.Series(
    {
        "module": (
            "Module 10 • Advanced Applications"
        ),
        "lesson": (
            "Lesson 60 • Conversational AI and Dialogue Systems"
        ),
        "intent_examples": len(
            intent_frame
        ),
        "intents": len(
            intent_model.classes_
        ),
        "state_slots": len(
            REQUIRED_BOOKING_SLOTS
        ),
        "memory_size": 6,
        "seed": SEED,
        "offline_execution": True,
        "python": (
            platform.python_version()
        ),
    },
    name="Lesson 60 experiment",
)

# 53. Knowledge Check

1. What distinguishes task-oriented from open-domain dialogue?
2. What is intent classification?
3. What is slot filling?
4. What does dialogue state track?
5. What does the dialogue policy decide?
6. Why is confirmation important?
7. When should a system clarify?
8. What is fallback behavior?
9. What is abstention?
10. Why should short-term and long-term memory be separated?
11. What is task success?
12. What is turn efficiency?
13. Why can state errors be more serious than response-style errors?
14. Why must Arabic dialogue define a tashkeel policy?
15. What should a production dialogue system monitor?

# 54. Exercises

1. Add more intents.
2. Add entity extraction for booking IDs.
3. Add a confirmation state.
4. Add cancellation state transitions.
5. Add a weather tool stub.
6. Add multilingual intent examples.
7. Add fully vocalized Arabic training examples.
8. Add code-switching examples.
9. Build a dialogue-level evaluation dataset.
10. Create a final evaluation table with intent accuracy, slot accuracy, state accuracy, task success, turn efficiency, fallback rate, and latency.

## Challenge Exercises

1. Replace TF-IDF intent classification with a Transformer encoder.
2. Add a learned dialogue policy.
3. Add retrieval-grounded conversational QA.
4. Add tool calling with explicit confirmation for side effects.
5. Build a multilingual conversational assistant with structured state and memory.

# 55. Summary and Next Lesson

In this lesson:

- task-oriented and open-domain dialogue were distinguished;
- intent classification and slot extraction were implemented;
- dialogue state tracking preserved information across turns;
- policy rules selected next actions;
- response templates generated context-aware replies;
- short-term memory and session state were separated;
- confirmation, clarification, fallback, and abstention were incorporated;
- intent accuracy, slot accuracy, state accuracy, task success, and turn efficiency were evaluated;
- multilingual and Arabic/tashkeel considerations were included;
- privacy, safety, latency, and production monitoring were connected to system design.

## Next Lesson

**Lesson 61: Advanced Multimodal NLP — Vision-Language Models, Document Understanding,
OCR-Aware Reasoning, and Multimodal Retrieval**

# References

- Jurafsky, D. and Martin, J. work on dialogue systems.
- Young, S. et al. work on statistical spoken dialogue systems.
- Henderson, M. et al. work on dialogue state tracking.
- Budzianowski, P. et al. work on MultiWOZ.
- Research on neural task-oriented dialogue, LLM agents, and conversational evaluation.